In [2]:
import pandas as pd
from datetime import datetime, date, time, timedelta
from pathlib import Path


# ============================================================
# CONFIGURAZIONE
# ============================================================
INPUT_FILE = "risposte_form.xlsx"      # <-- metti qui il nome del file Excel del form
OUTPUT_FILE = "scheduling_output.xlsx"

START_DATE = date(2026, 4, 17)
END_DATE   = date(2026, 4, 30)

EXCLUDED_DATES = {
    date(2026, 4, 25),   # festività esplicitamente esclusa
}

EXCLUDED_FULL_NAMES = {
    "alessandro genua",
}

# Slot da 90 minuti compatibili con le fasce richieste
MORNING_SLOTS = [
    (time(9, 0),  time(10, 30)),
    (time(10, 30), time(12, 0)),
]

AFTERNOON_SLOTS = [
    (time(14, 0), time(15, 30)),
    (time(15, 30), time(17, 0)),
]

DAY_NAME_MAP = {
    "lunedi": "Monday",
    "lunedì": "Monday",
    "martedi": "Tuesday",
    "martedì": "Tuesday",
    "mercoledi": "Wednesday",
    "mercoledì": "Wednesday",
    "giovedi": "Thursday",
    "giovedì": "Thursday",
    "venerdi": "Friday",
    "venerdì": "Friday",
    "sabato": "Saturday",
    "domenica": "Sunday",
}

VALID_WEEKDAYS = {"Monday", "Tuesday", "Wednesday", "Thursday", "Friday"}


# ============================================================
# FUNZIONI UTILI
# ============================================================
def normalize_text(x):
    """Converte in stringa pulita e minuscola."""
    if pd.isna(x):
        return ""
    return str(x).strip().lower()


def build_full_name(row):
    """
    Costruisce il nome completo usando Nome1 + Cognome se disponibili,
    altrimenti prova con Nome.
    """
    nome = str(row.get("Nome1", "")).strip()
    cognome = str(row.get("Cognome", "")).strip()

    full_name = f"{nome} {cognome}".strip()

    if full_name:
        return full_name

    return str(row.get("Nome", "")).strip()


def parse_days(days_str):
    """
    Converte la stringa 'Lunedi;Martedi;...' in un insieme di weekday inglesi.
    """
    s = normalize_text(days_str)
    if not s:
        return set()

    parts = [p.strip() for p in s.split(";") if p.strip()]
    mapped = set()

    for p in parts:
        if p in DAY_NAME_MAP:
            mapped.add(DAY_NAME_MAP[p])

    return mapped


def parse_time_bands(bands_str):
    """
    Riconosce le fasce 'Mattina...' e 'Pomeriggio...'
    """
    s = normalize_text(bands_str)
    bands = set()

    if "mattina" in s:
        bands.add("morning")
    if "pomeriggio" in s:
        bands.add("afternoon")

    return bands


def daterange(start_date, end_date):
    """Genera tutte le date tra start_date ed end_date inclusi."""
    current = start_date
    while current <= end_date:
        yield current
        current += timedelta(days=1)


def is_working_day(d):
    """
    Giorni lavorativi:
    - lunedì-venerdì
    - escluso 25 aprile
    - esclusi sabato e domenica
    """
    if d in EXCLUDED_DATES:
        return False
    if d.weekday() >= 5:  # 5=sabato, 6=domenica
        return False
    return True


def generate_slots():
    """
    Genera tutti gli slot disponibili nel periodo richiesto.
    Ogni slot è un dict con data, giorno della settimana, fascia, orari.
    """
    slots = []
    slot_id = 1

    for d in daterange(START_DATE, END_DATE):
        if not is_working_day(d):
            continue

        weekday_name = d.strftime("%A")

        for start_t, end_t in MORNING_SLOTS:
            slots.append({
                "slot_id": slot_id,
                "date": d,
                "weekday": weekday_name,
                "band": "morning",
                "start_time": start_t,
                "end_time": end_t,
                "assigned": False,
                "assigned_to": None
            })
            slot_id += 1

        for start_t, end_t in AFTERNOON_SLOTS:
            slots.append({
                "slot_id": slot_id,
                "date": d,
                "weekday": weekday_name,
                "band": "afternoon",
                "start_time": start_t,
                "end_time": end_t,
                "assigned": False,
                "assigned_to": None
            })
            slot_id += 1

    return slots


def load_participants(excel_file):
    """
    Legge il file Excel e prepara i dati dei partecipanti.
    Esclude Alessandro Genua.
    """
    df = pd.read_excel(excel_file)

    required_columns = ["Nome1", "Cognome", "Giorni preferiti", "Fascia oraria"]
    missing = [c for c in required_columns if c not in df.columns]
    if missing:
        raise ValueError(f"Colonne mancanti nel file Excel: {missing}")

    participants = []

    for idx, row in df.iterrows():
        full_name = build_full_name(row)
        full_name_norm = normalize_text(full_name)

        if full_name_norm in EXCLUDED_FULL_NAMES:
            continue

        preferred_days = parse_days(row.get("Giorni preferiti", ""))
        preferred_bands = parse_time_bands(row.get("Fascia oraria", ""))

        # Se mancano preferenze, il soggetto viene tenuto ma probabilmente resterà non assegnato
        participants.append({
            "original_index": idx,
            "full_name": full_name,
            "email": row.get("E-mail", ""),
            "preferred_days": preferred_days,
            "preferred_bands": preferred_bands,
            "raw_days": row.get("Giorni preferiti", ""),
            "raw_bands": row.get("Fascia oraria", "")
        })

    return participants


def count_compatible_slots(participant, slots):
    """Conta quanti slot sono compatibili con le preferenze del partecipante."""
    count = 0
    for slot in slots:
        if slot["weekday"] in participant["preferred_days"] and slot["band"] in participant["preferred_bands"]:
            count += 1
    return count


def assign_slots(participants, slots):
    """
    Assegnazione greedy:
    - prima i soggetti più 'rigidi' (meno slot compatibili)
    - poi il primo slot libero compatibile in ordine cronologico
    """
    # Ordina i partecipanti dal più difficile da collocare al più facile
    participants_sorted = sorted(
        participants,
        key=lambda p: (
            count_compatible_slots(p, slots),
            p["full_name"].lower()
        )
    )

    assigned = []
    unassigned = []

    for p in participants_sorted:
        chosen_slot = None

        for slot in slots:
            if slot["assigned"]:
                continue

            if slot["weekday"] in p["preferred_days"] and slot["band"] in p["preferred_bands"]:
                chosen_slot = slot
                break

        if chosen_slot is not None:
            chosen_slot["assigned"] = True
            chosen_slot["assigned_to"] = p["full_name"]

            assigned.append({
                "Nome e Cognome": p["full_name"],
                "E-mail": p["email"],
                "Giorni preferiti": p["raw_days"],
                "Fascia oraria": p["raw_bands"],
                "Data": chosen_slot["date"].strftime("%d/%m/%Y"),
                "Giorno": chosen_slot["weekday"],
                "Ora inizio": chosen_slot["start_time"].strftime("%H:%M"),
                "Ora fine": chosen_slot["end_time"].strftime("%H:%M"),
            })
        else:
            unassigned.append({
                "Nome e Cognome": p["full_name"],
                "E-mail": p["email"],
                "Giorni preferiti": p["raw_days"],
                "Fascia oraria": p["raw_bands"],
                "Motivo": "Nessuno slot compatibile disponibile"
            })

    return assigned, unassigned


def save_results(assigned, unassigned, slots, output_file):
    """Salva i risultati in un file Excel con più fogli."""
    df_assigned = pd.DataFrame(assigned)
    df_unassigned = pd.DataFrame(unassigned)

    all_slots = []
    for s in slots:
        all_slots.append({
            "slot_id": s["slot_id"],
            "Data": s["date"].strftime("%d/%m/%Y"),
            "Giorno": s["weekday"],
            "Fascia": "Mattina" if s["band"] == "morning" else "Pomeriggio",
            "Ora inizio": s["start_time"].strftime("%H:%M"),
            "Ora fine": s["end_time"].strftime("%H:%M"),
            "Assegnato": "Sì" if s["assigned"] else "No",
            "Partecipante": s["assigned_to"] if s["assigned_to"] else ""
        })

    df_slots = pd.DataFrame(all_slots)

    with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
        df_assigned.to_excel(writer, sheet_name="Schedulati", index=False)
        df_unassigned.to_excel(writer, sheet_name="Non_schedulati", index=False)
        df_slots.to_excel(writer, sheet_name="Tutti_gli_slot", index=False)


def main():
    input_path = Path(INPUT_FILE)

    if not input_path.exists():
        raise FileNotFoundError(
            f"File Excel non trovato: {INPUT_FILE}\n"
            f"Metti il file nella stessa cartella dello script oppure modifica INPUT_FILE."
        )

    participants = load_participants(INPUT_FILE)
    slots = generate_slots()
    assigned, unassigned = assign_slots(participants, slots)
    save_results(assigned, unassigned, slots, OUTPUT_FILE)

    print("==========================================")
    print("Scheduling completato")
    print(f"Partecipanti letti: {len(participants)}")
    print(f"Soggetti schedulati: {len(assigned)}")
    print(f"Soggetti non schedulati: {len(unassigned)}")
    print(f"Output salvato in: {OUTPUT_FILE}")
    print("==========================================")

    if unassigned:
        print("\nPartecipanti non schedulati:")
        for u in unassigned:
            print(f"- {u['Nome e Cognome']} | {u['Motivo']}")


if __name__ == "__main__":
    main()

FileNotFoundError: File Excel non trovato: risposte_form.xlsx
Metti il file nella stessa cartella dello script oppure modifica INPUT_FILE.